# S10 - Conditional Expressions and Procedures

## Libraries

In [1]:
import os
import sqlite3 as sql
import pandas as pd

## Functions

In [2]:
# Function taking a SQL query file to format its text for Jupyter Notebook
def query_text_format(path, file):
    # Change directory
    try:
        os.chdir(path)
    except FileNotFoundError:
        print(f"El directorio {path} no se encontró.")
        return
    
    # Read file
    try:
        with open(file, 'r') as f:
            query_log = f.read()
    except FileNotFoundError:
        print(f"El archivo {file} no se encontró.")
        return
    
    # Text formatting
    # Removal of SQL comment closing
    query_log = query_log.replace(' */', '')
    # Split into lines
    query_log_lines = query_log.split('\n')
    
    # Enumerated iteration over all file lines
    for i, line in enumerate(query_log_lines):
        query_log_lines[i] = line.strip()
        
        # Replacement of SQL comment opening
        if query_log_lines[i][:2] == '/*':
            # Search two integers to detect lesson start
            double_int = []            
            for char in line[3:5]:
                try:
                    if isinstance(int(char), int):
                        double_int.append(1)
                except:
                    pass
            
            # If two integers found, replacement according to lesson start
            if len(double_int) == 2:
                query_log_lines[i] = query_log_lines[i].replace('/*', '###')
            # If a letter, replacement according to assessment exercise start
            elif query_log_lines[i][3].isalpha() and query_log_lines[i][4] == '.':
                query_log_lines[i] = query_log_lines[i].replace('/*', '####')
            # Remaining SQL comments are Python comments, too
            else:
                query_log_lines[i] = query_log_lines[i].replace('/*', '#')
        
        # If line not empty
        elif line != '':
            query_log_lines[i] = '\t' + query_log_lines[i] + ' \\'
        
        # Rest of cases (potential)
        else:
            pass
        
    return '\n'.join(query_log_lines)

In [3]:
# Function taking the SQL queries and building Python code to execute them
def build_python_queries(text):
    # Split into lines
    query_lines = text.split('\n')
    
    # Build SQL queries in Python
    resulting_text = []
    flg_first_sql_line = True
    
    for i, line in enumerate(query_lines):
        # Empty line
        if line == '':
            resulting_text.append(line)
        # Line starting with tab
        elif query_lines[i][0] == '\t':
            # First query line
            if flg_first_sql_line:
                # First query line in one line query
                if (i + 1 <= len(query_lines) - 1) and ((query_lines[i + 1] == '') or ((query_lines[i + 1][0] != '') and (query_lines[i + 1][0] == '#'))):
                    resulting_text.append('query = " \\')
                    resulting_text.append(line)
                    resulting_text.append('\t"')
                    resulting_text.append('execute_query(query, conn)')
                # First query line in multiple lines query
                else:
                    resulting_text.append('query = " \\')
                    resulting_text.append(line)
                    flg_first_sql_line = False
            # Second, or other, query line
            else:
                # Last query line
                if (i + 1 <= len(query_lines) - 1) and ((query_lines[i + 1] == '') or ((query_lines[i + 1][0] != '') and (query_lines[i + 1][0] == '#'))):
                    resulting_text.append(line)
                    resulting_text.append('\t"')
                    resulting_text.append('execute_query(query, conn)')
                    flg_first_sql_line = True
                # Query line, absolute last one
                elif i == len(query_lines) - 1:
                    resulting_text.append(line)
                    resulting_text.append('\t"')
                    resulting_text.append('execute_query(query, conn)')
                # Query lines that are not first nor last
                else:
                    resulting_text.append(line)
        # Other lines
        else:
            resulting_text.append(line)
    
    return '\n'.join(resulting_text)

In [4]:
# Function taking the SQL query text and executing it
def execute_query(query_text, connection):
    try:
        # Check if the query is a SELECT statement
        if query_text.strip().upper().startswith("SELECT"):
            # DataFrame from query
            results_df = pd.read_sql_query(
                query_text,
                connection
            )
            print(results_df)
        else:
            # Execute queries that do not return results
            with connection:
                connection.execute(query_text)
            print("Query executed successfully.")
    except Exception as e:
        print(f"Error executing the query: {e}")

## Settings

In [5]:
# Limit removal for showing pandas.DataFrames' columns
pd.set_option('display.max_columns', None)
# Limit removal for showing pandas.DataFrames' rows
pd.set_option('display.max_rows', None)
# Modification of console with for displaying
pd.set_option('display.width', 8000)

## SQL Query Formatting

In [6]:
# First formatting
path = 'G:\\15_Estudio\\Udemy\\SQL - Portilla Complete Bootcamp'
file = 'UDM-SQL-BTCMP--010.sql'

first_pass = query_text_format(path, file)
print(first_pass)

### 72. Conditional Expressions and Procedures - Introduction
# Blank/In notes

### 73. CASE
# Exploration, table customer
	SELECT * \
	FROM customer \
	LIMIT 1; \

# General CASE
# Customer tiers: 1 to 100, Premium; 101 to 200, Plus; 201 and up, regular
	SELECT \
	customer_id, \
	CASE \
	WHEN customer_id BETWEEN 1 AND 100 THEN 'Premium' \
	WHEN customer_id BETWEEN 101 AND 200 THEN 'Plus' \
	ELSE 'Regular' \
	END AS customer_class \
	FROM customer \
	ORDER BY customer_id ASC; \

# CASE Expression
# Raffle where winner is customer_id = 2, and second place
# is customer_id = 5; the rest is normal
	SELECT \
	customer_id, \
	CASE customer_id \
	WHEN 2 THEN 'Winner' \
	WHEN 5 THEN 'Second Place' \
	ELSE 'Normal' \
	END AS raffle_result \
	FROM customer \
	ORDER BY customer_id ASC; \

# Exploration, film table
	SELECT * \
	FROM film \
	LIMIT 1; \

# Exploration, rental_rate unique valuse
	SELECT DISTINCT(rental_rate) \
	FROM film; \

# How many of each rental_rate? Using CASE
# First, explor

In [7]:
# Final formatting
working_text = build_python_queries(first_pass)
print(working_text)

### 72. Conditional Expressions and Procedures - Introduction
# Blank/In notes

### 73. CASE
# Exploration, table customer
query = " \
	SELECT * \
	FROM customer \
	LIMIT 1; \
	"
execute_query(query, conn)

# General CASE
# Customer tiers: 1 to 100, Premium; 101 to 200, Plus; 201 and up, regular
query = " \
	SELECT \
	customer_id, \
	CASE \
	WHEN customer_id BETWEEN 1 AND 100 THEN 'Premium' \
	WHEN customer_id BETWEEN 101 AND 200 THEN 'Plus' \
	ELSE 'Regular' \
	END AS customer_class \
	FROM customer \
	ORDER BY customer_id ASC; \
	"
execute_query(query, conn)

# CASE Expression
# Raffle where winner is customer_id = 2, and second place
# is customer_id = 5; the rest is normal
query = " \
	SELECT \
	customer_id, \
	CASE customer_id \
	WHEN 2 THEN 'Winner' \
	WHEN 5 THEN 'Second Place' \
	ELSE 'Normal' \
	END AS raffle_result \
	FROM customer \
	ORDER BY customer_id ASC; \
	"
execute_query(query, conn)

# Exploration, film table
query = " \
	SELECT * \
	FROM film \
	LIMIT 1; \
	"
execut

This text will be used to create the whole of the Practice section in this notebook.

## Practice

### Connection with Data Base

In [8]:
# Connection and cursor
conn = sql.connect('G:\\15_Estudio\\Udemy\\SQL - Portilla Complete Bootcamp\\dvdrental.db')
cur = conn.cursor()

### 72. Conditional Expressions and Procedures - Introduction

In [9]:
# Blank/In notes

### 73. CASE

In [10]:
# Exploration, table customer
query = " \
	SELECT * \
	FROM customer \
	LIMIT 1; \
	"
execute_query(query, conn)

   customer_id  store_id first_name last_name                         email  address_id activebool create_date              last_update  active
0          524         1      Jared       Ely  jared.ely@sakilacustomer.org         530          t  2006-02-14  2013-05-26 14:49:45.738       1


In [11]:
# General CASE
# Customer tiers: 1 to 100, Premium; 101 to 200, Plus; 201 and up, regular
query = " \
	SELECT \
	customer_id, \
	CASE \
	WHEN customer_id BETWEEN 1 AND 100 THEN 'Premium' \
	WHEN customer_id BETWEEN 101 AND 200 THEN 'Plus' \
	ELSE 'Regular' \
	END AS customer_class \
	FROM customer \
	ORDER BY customer_id ASC; \
	"
execute_query(query, conn)

     customer_id customer_class
0              1        Premium
1              2        Premium
2              3        Premium
3              4        Premium
4              5        Premium
5              6        Premium
6              7        Premium
7              8        Premium
8              9        Premium
9             10        Premium
10            11        Premium
11            12        Premium
12            13        Premium
13            14        Premium
14            15        Premium
15            16        Premium
16            17        Premium
17            18        Premium
18            19        Premium
19            20        Premium
20            21        Premium
21            22        Premium
22            23        Premium
23            24        Premium
24            25        Premium
25            26        Premium
26            27        Premium
27            28        Premium
28            29        Premium
29            30        Premium
30      

In [12]:
# CASE Expression
# Raffle where winner is customer_id = 2, and second place
# is customer_id = 5; the rest is normal
query = " \
	SELECT \
	customer_id, \
	CASE customer_id \
	WHEN 2 THEN 'Winner' \
	WHEN 5 THEN 'Second Place' \
	ELSE 'Normal' \
	END AS raffle_result \
	FROM customer \
	ORDER BY customer_id ASC; \
	"
execute_query(query, conn)

     customer_id raffle_result
0              1        Normal
1              2        Winner
2              3        Normal
3              4        Normal
4              5  Second Place
5              6        Normal
6              7        Normal
7              8        Normal
8              9        Normal
9             10        Normal
10            11        Normal
11            12        Normal
12            13        Normal
13            14        Normal
14            15        Normal
15            16        Normal
16            17        Normal
17            18        Normal
18            19        Normal
19            20        Normal
20            21        Normal
21            22        Normal
22            23        Normal
23            24        Normal
24            25        Normal
25            26        Normal
26            27        Normal
27            28        Normal
28            29        Normal
29            30        Normal
30            31        Normal
31      

In [13]:
# Exploration, film table
query = " \
	SELECT * \
	FROM film \
	LIMIT 1; \
	"
execute_query(query, conn)

   film_id            title                                        description  release_year  language_id  rental_duration  rental_rate  length  replacement_cost rating              last_update special_features                                           fulltext
0      133  Chamber Italian  A Fateful Reflection of a Moose And a Husband ...          2006            1                7         4.99     117             14.99  NC-17  2013-05-26 14:50:58.951       {Trailers}  'chamber':1 'fate':4 'husband':11 'italian':2 ...


In [14]:
# Exploration, rental_rate unique valuse
query = " \
	SELECT DISTINCT(rental_rate) \
	FROM film; \
	"
execute_query(query, conn)

   rental_rate
0         4.99
1         0.99
2         2.99


In [15]:
# How many of each rental_rate? Using CASE
# First, explore 0.99 yes/no
query = " \
	SELECT \
	rental_rate, \
	CASE rental_rate \
	WHEN 0.99 THEN 1 \
	ELSE 0 \
	END AS something \
	FROM film; \
	"
execute_query(query, conn)

     rental_rate  something
0           4.99          0
1           4.99          0
2           4.99          0
3           4.99          0
4           0.99          1
5           4.99          0
6           2.99          0
7           2.99          0
8           2.99          0
9           2.99          0
10          4.99          0
11          2.99          0
12          4.99          0
13          0.99          1
14          0.99          1
15          0.99          1
16          4.99          0
17          0.99          1
18          2.99          0
19          2.99          0
20          0.99          1
21          0.99          1
22          0.99          1
23          4.99          0
24          4.99          0
25          2.99          0
26          0.99          1
27          2.99          0
28          2.99          0
29          0.99          1
30          0.99          1
31          4.99          0
32          2.99          0
33          2.99          0
34          4.99    

In [16]:
# Count 0.99 as bargain
query = " \
	SELECT \
	SUM(CASE rental_rate \
	WHEN 0.99 THEN 1 \
	ELSE 0 \
	END) AS tot_qty_bargain \
	FROM film; \
	"
execute_query(query, conn)

   tot_qty_bargain
0              341


In [17]:
# Count 2.99 as cheap
query = " \
	SELECT \
	SUM(CASE rental_rate \
	WHEN 2.99 THEN 1 \
	ELSE 0 \
	END) AS tot_qty_cheap \
	FROM film; \
	"
execute_query(query, conn)

   tot_qty_cheap
0            323


In [18]:
# Count 4.99 as costly
query = " \
	SELECT \
	SUM(CASE rental_rate \
	WHEN 4.99 THEN 1 \
	ELSE 0 \
	END) AS tot_qty_costly \
	FROM film; \
	"
execute_query(query, conn)

   tot_qty_costly
0             336


In [19]:
# All of the three in only one query
query = " \
	SELECT \
	SUM(CASE rental_rate \
	WHEN 0.99 THEN 1 \
	ELSE 0 \
	END) AS tot_qty_bargain, \
	SUM(CASE rental_rate \
	WHEN 2.99 THEN 1 \
	ELSE 0 \
	END) AS tot_qty_cheap, \
	SUM(CASE rental_rate \
	WHEN 4.99 THEN 1 \
	ELSE 0 \
	END) AS tot_qty_costly \
	FROM film; \
	"
execute_query(query, conn)
# It allows for results in columns, which is difficult to do
# using other tools previously learned. Also, it allows to
# apply functions on the results of the cases

   tot_qty_bargain  tot_qty_cheap  tot_qty_costly
0              341            323             336


### 74. Challenge: CASE

#### a. How many films per film rating (limited to R, PG, and PG-13)?

In [20]:
# Explore film table
query = " \
	SELECT * \
	FROM film \
	LIMIT 1; \
	"
execute_query(query, conn)

   film_id            title                                        description  release_year  language_id  rental_duration  rental_rate  length  replacement_cost rating              last_update special_features                                           fulltext
0      133  Chamber Italian  A Fateful Reflection of a Moose And a Husband ...          2006            1                7         4.99     117             14.99  NC-17  2013-05-26 14:50:58.951       {Trailers}  'chamber':1 'fate':4 'husband':11 'italian':2 ...


In [21]:
# Explore, unique ratings?
query = " \
	SELECT DISTINCT(rating) \
	FROM film; \
	"
execute_query(query, conn)

  rating
0  NC-17
1      R
2  PG-13
3     PG
4      G


In [22]:
# Count films per rating
query = " \
	SELECT \
        SUM( \
            CASE rating \
                WHEN 'PG-13' THEN 1 \
                ELSE 0 \
            END \
        ) AS \"PG-13\", \
        SUM( \
            CASE rating \
                WHEN 'NC-17' THEN 1 \
                ELSE 0 \
            END \
        ) AS \"NC-17\", \
        SUM( \
            CASE rating \
                WHEN 'R' THEN 1 \
                ELSE 0 \
            END \
        ) AS \"R\", \
        SUM( \
            CASE rating \
                WHEN 'G' THEN 1 \
                ELSE 0 \
            END \
        ) AS \"G\", \
        SUM( \
            CASE rating \
                WHEN 'PG' THEN 1 \
                ELSE 0 \
            END \
        ) AS \"PG\" \
	FROM film; \
	"
execute_query(query, conn)
# In Postgre, there was a comment after AS \"PG-13\", and it was
# -- Double quoutes, mandatory
# but SQLite does not allow for comments in the middle of queries

   PG-13  NC-17    R    G   PG
0    223    210  195  178  194


### 75. COALESCE

In [23]:
# Blank/In notes

### 76. CAST

In [24]:
# Function
query = " \
	SELECT CAST('5' AS INTEGER); \
	"
execute_query(query, conn)

   CAST('5' AS INTEGER)
0                     5


In [25]:
# PostgreSQL operator
#query = " \
#	SELECT '5'::INTEGER; \
#	"
#execute_query(query, conn)

# Commented out, as it doesn't work in SQLite

In [26]:
# Explore rental table
query = " \
	SELECT * \
	FROM rental \
	LIMIT 1; \
	"
execute_query(query, conn)

   rental_id          rental_date  inventory_id  customer_id          return_date  staff_id          last_update
0          2  2005-05-24 22:54:33          1525          459  2005-05-28 19:40:33         1  2006-02-16 02:30:53


In [27]:
# Count the number of digits in each inventory_id
query = " \
	SELECT inventory_id, LENGTH(CAST(inventory_id AS VARCHAR)) \
	FROM rental \
	LIMIT 15; \
	"
execute_query(query, conn)

    inventory_id  LENGTH(CAST(inventory_id AS VARCHAR))
0           1525                                      4
1           1711                                      4
2           2452                                      4
3           2079                                      4
4           2792                                      4
5           3995                                      4
6           2346                                      4
7           2580                                      4
8           1824                                      4
9           4443                                      4
10          1584                                      4
11          2294                                      4
12          2701                                      4
13          3049                                      4
14           389                                      3


### 77. NULLIF

In [28]:
# Blank/In notes

### 78. Views

In [29]:
# Explore customer, address tables
query = " \
	SELECT * \
	FROM customer \
	LIMIT 1; \
	"
execute_query(query, conn)

   customer_id  store_id first_name last_name                         email  address_id activebool create_date              last_update  active
0          524         1      Jared       Ely  jared.ely@sakilacustomer.org         530          t  2006-02-14  2013-05-26 14:49:45.738       1


In [30]:
query = " \
	SELECT * \
	FROM address \
	LIMIT 1; \
	"
execute_query(query, conn)

   address_id            address address2 district  city_id postal_code phone          last_update
0           1  47 MySakila Drive     None  Alberta      300        None  None  2006-02-15 09:45:30


In [31]:
# Join tables
query = " \
	SELECT first_name, last_name, address \
	FROM customer cu \
	JOIN address ad \
	ON cu.address_id = ad.address_id; \
	"
execute_query(query, conn)

      first_name     last_name                                 address
0          Jared           Ely                 1003 Qinhuangdao Street
1           Mary         Smith                          1913 Hanoi Way
2       Patricia       Johnson                        1121 Loja Avenue
3          Linda      Williams                       692 Joliet Street
4        Barbara         Jones                        1566 Inegl Manor
5      Elizabeth         Brown                         53 Idfu Parkway
6       Jennifer         Davis         1795 Santiago de Compostela Way
7          Maria        Miller      900 Santiago de Compostela Parkway
8          Susan        Wilson                          478 Joliet Way
9       Margaret         Moore                       613 Korolev Drive
10       Dorothy        Taylor                          1531 Sal Drive
11          Lisa      Anderson                     1542 Tarlac Parkway
12         Nancy        Thomas                        808 Bhopal Manor
13    

In [32]:
# Assuming this query will be repeatedly used, sava as view
query = " \
	CREATE VIEW customer_info AS \
	SELECT first_name, last_name, address \
	FROM customer cu \
	JOIN address ad \
	ON cu.address_id = ad.address_id; \
	"
execute_query(query, conn)

Query executed successfully.


In [33]:
# Verify view creation, explore customer_info
query = " \
	SELECT * \
	FROM customer_info \
	LIMIT 1; \
	"
execute_query(query, conn)

  first_name last_name                  address
0      Jared       Ely  1003 Qinhuangdao Street


In [34]:
# Alter/update the view

# In Postgre, query was:

#	CREATE OR REPLACE VIEW customer_info AS \
#	SELECT cu.first_name, cu.last_name, ad.address, ad.district \
#	FROM customer cu \
#	JOIN address ad \
#	ON cu.address_id = ad.address_id; \

# But in SQLite, OR REPLACE not supported for views.
# Instead:

# Step 1: Drop the existing view if it exists
drop_view_query = "DROP VIEW IF EXISTS customer_info;"
execute_query(drop_view_query, conn)

# Step 2: Create the new view
create_view_query = " \
    CREATE VIEW customer_info AS \
    SELECT cu.first_name, cu.last_name, ad.address, ad.district \
    FROM customer cu \
    JOIN address ad \
    ON cu.address_id = ad.address_id; \
    "
execute_query(create_view_query, conn)

Query executed successfully.
Query executed successfully.


In [35]:
# Rename the view

# In Postgre the query was:

#	ALTER VIEW customer_info \
#	RENAME TO c_info; \


# But in SQLite it has to be done differently:

# Step 1: Drop the existing view if it exists
drop_view_query = "DROP VIEW IF EXISTS customer_info;"
execute_query(drop_view_query, conn)

# Step 2: Create the new view with the new name
create_view_query = " \
    CREATE VIEW c_info AS \
    SELECT cu.first_name, cu.last_name, ad.address, ad.district \
    FROM customer cu \
    JOIN address ad \
    ON cu.address_id = ad.address_id; \
    "
execute_query(create_view_query, conn)

Query executed successfully.
Query executed successfully.


In [36]:
# Verify new view name, explore c_info
query = " \
	SELECT * \
	FROM c_info \
	LIMIT 1; \
	"
execute_query(query, conn)

  first_name last_name                  address   district
0      Jared       Ely  1003 Qinhuangdao Street  West Java


In [37]:
# Drop the view
# DROP VIEW c_info;
query = " \
	DROP VIEW IF EXISTS c_info; \
	"
execute_query(query, conn)

Query executed successfully.
